# PTB-XL 전처리 — 설명과 코드를 함께 읽는 노트북

이 노트북 하나에 **전처리 설명, 실행 코드, 파형 비교, 데이터 저장·검증**을 모았습니다.
별도 Python 파일을 import할 필요 없이 위에서 아래로 실행하면 됩니다.

**처리 흐름:** 500 Hz 원본 → 250 Hz 리샘플링 → 0.67–50 Hz 필터 → ECG·lead별 z-score → split별 저장.

이미 전처리한 21,799개 ECG는 `artifacts/ptbxl_250hz`에 있습니다.
기본 설정은 기존 산출물을 읽어 검증하므로 전체 전처리를 다시 기다릴 필요가 없습니다.
설정을 바꾸어 새 데이터를 생성하려면 아래 설정 셀의 `OUTPUT_DIR`을 새 경로로 바꾸세요.

> 전처리 설정은 이전 로컬 EDA의 권고안을 사용했습니다. 노션 보고서 원문과의 일치 여부는 아직 확인하지 못했습니다.
> 필터·정규화는 연구 선택이므로 downstream 성능 비교를 통해 평가해야 합니다.

2026-10-24: 기본 band-pass를 0.67–50 Hz로 변경했습니다. 기존 40 Hz 산출물은 보존하고, 원본 500 Hz 파형에서 별도 폴더에 새로 만듭니다. 터미널에서는 `python preprocessing/run_preprocessing.py`로 같은 처리 코드를 실행할 수 있습니다.


## 1. 실행 환경과 라이브러리

NumPy는 배열, pandas는 메타데이터, WFDB는 ECG 파일 읽기, SciPy는 신호 처리,
Matplotlib은 파형 비교에 사용합니다.

현재 저장소의 `.venv`에는 이 라이브러리가 설치되어 있습니다.
Jupyter/VS Code에서 **해당 가상환경의 Python**을 커널로 선택하세요.
커널 구성이나 라이브러리 설치가 필요하면 다음 명령을 터미널에서 실행합니다.

```powershell
.\.venv\Scripts\python.exe -m pip install numpy pandas scipy wfdb matplotlib ipykernel
.\.venv\Scripts\python.exe -m ipykernel install --user --name ecg-preprocessing --display-name "ECG preprocessing"
```

다른 환경에서는 다음 설치 셀의 주석을 해제해 현재 커널에 설치할 수 있습니다.


In [ ]:
# 필요한 경우에만 주석을 해제해 실행하세요.
# %pip install numpy pandas scipy wfdb matplotlib ipykernel


In [ ]:
import ast
import hashlib
import json
import math
import time
import uuid
from datetime import datetime, timezone
from fractions import Fraction
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from numpy.lib.format import open_memmap
from scipy.signal import butter, resample_poly, sosfiltfilt
from IPython.display import display

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})
pd.set_option("display.max_columns", 20)
print("ECG preprocessing libraries loaded.")


## 2. 경로와 전처리 설정

노트북을 `preprocessing/`, 저장소 루트 또는 상위 작업 폴더에서 열어도 경로를 찾습니다.
데이터를 다른 위치에 옮겼다면 `DATA_DIR`만 수정하세요.

| 설정 | 기본값 | 의미 |
| --- | --- | --- |
| 원본 / 출력 주파수 | 500 / 250 Hz | 10초 × 250 Hz = 2,500 samples |
| 필터 | 0.67–50 Hz, 4차 Butterworth | baseline wander·고주파 성분 억제 |
| 정규화 | per-record | ECG마다 lead별 평균 0, 표준편차 1 |
| 라벨 없는 기록 | 유지 | SSL에 사용, 지도학습에서는 validity mask 적용 |
| 실행 범위 | full | 전체 21,799개; smoke는 split별 처음 4개 |

`NORMALIZATION`은 `"per-record"`, `"per-lead-train"`, `"none"` 중 선택합니다.
`USE_BANDPASS=False`는 필터 효과 비교용입니다.
`RUN_MODE="smoke"`는 빠른 저장 기능 점검용으로, 대표 표본은 아닙니다.


In [ ]:
# 실행 위치에 관계없이 저장소와 데이터 경로 찾기
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
REPO_DIR = next(
    (p for p in candidates if p.name == "predictability-guided-ecg-masking"),
    None,
)
if REPO_DIR is None:
    REPO_DIR = next(
        (p / "predictability-guided-ecg-masking" for p in candidates
         if (p / "predictability-guided-ecg-masking" / "preprocessing").is_dir()),
        None,
    )
if REPO_DIR is None:
    raise FileNotFoundError("Open this notebook inside the repository or its parent folder.")

DATA_DIR = REPO_DIR.parent / "ptb-xl" / "ptb-xl"
RUN_MODE = "full"                         # "full" 또는 "smoke"
OUTPUT_DIR = REPO_DIR / "preprocessing" / "artifacts" / (
    "ptbxl_250hz_bp067_50" if RUN_MODE == "full" else "ptbxl_250hz_bp067_50_smoke"
)

SOURCE_FS = 500
TARGET_FS = 250
USE_BANDPASS = True
LOW_HZ, HIGH_HZ = 0.67, 50.0
FILTER_ORDER = 4
NORMALIZATION = "per-record"
EPS = 1e-6
DROP_UNLABELED = False
LIMIT_PER_SPLIT = None if RUN_MODE == "full" else 4
PROGRESS_EVERY = 500

CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
SPLITS = ["train", "validation", "test"]
SOURCE_COLUMN = {100: "filename_lr", 500: "filename_hr"}[SOURCE_FS]

assert RUN_MODE in {"full", "smoke"}
assert NORMALIZATION in {"per-record", "per-lead-train", "none"}
assert 0 < TARGET_FS <= SOURCE_FS
assert EPS > 0
assert LIMIT_PER_SPLIT is None or LIMIT_PER_SPLIT > 0
assert not USE_BANDPASS or 0 < LOW_HZ < HIGH_HZ < TARGET_FS / 2

display(pd.DataFrame({
    "setting": ["dataset", "output", "sampling", "filter", "normalization", "run mode"],
    "value": [str(DATA_DIR), str(OUTPUT_DIR), f"{SOURCE_FS} -> {TARGET_FS} Hz",
              f"{LOW_HZ}-{HIGH_HZ} Hz" if USE_BANDPASS else "off",
              NORMALIZATION, RUN_MODE],
}))


## 3. 메타데이터 확인

`ptbxl_database.csv`의 한 행은 하나의 ECG입니다.

- `ecg_id`: 기록 식별자. 배열과 CSV를 연결할 때 사용합니다.
- `patient_id`: 환자 식별자. 같은 환자가 train/test에 섞이는지 검사합니다.
- `filename_hr/lr`: 각각 500/100 Hz WFDB 파일 경로.
- `scp_codes`: 진단·리듬 등의 SCP 코드 사전.
- `strat_fold`: 공식 분할 번호.

원본 파일은 읽기만 합니다. 나이 300은 익명화를 위한 값이므로 실제 300세로 해석하면 안 됩니다.
이 노트북은 나이 등 인구통계 변수를 모델 입력으로 가공하지 않습니다.


In [ ]:
for name in ["ptbxl_database.csv", "scp_statements.csv", f"records{SOURCE_FS}"]:
    if not (DATA_DIR / name).exists():
        raise FileNotFoundError(DATA_DIR / name)

database = pd.read_csv(DATA_DIR / "ptbxl_database.csv", low_memory=False)
statements = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
required = {"ecg_id", "patient_id", "scp_codes", "strat_fold", SOURCE_COLUMN}
assert required.issubset(database.columns), "Required metadata columns are missing."
assert database["ecg_id"].notna().all() and database["ecg_id"].is_unique
assert database["patient_id"].notna().all()

print(f"ECGs: {len(database):,} | patients: {database.patient_id.nunique():,}")
display(database[["ecg_id", "patient_id", "strat_fold", "scp_codes", SOURCE_COLUMN]].head())


## 4. 5개 진단 superclass를 multi-hot 라벨로 변환

| 열 순서 | 라벨 | 의미 |
| --- | --- | --- |
| 0 | NORM | 정상 |
| 1 | MI | 심근경색 |
| 2 | STTC | ST/T 변화 |
| 3 | CD | 전도 장애 |
| 4 | HYP | 비대 |

`scp_statements.csv`에서 diagnostic statement만 골라 superclass에 연결합니다.
예를 들어 NORM과 MI가 함께 있으면 `[1, 1, 0, 0, 0]`입니다.
여러 질환이 있는 ECG와 NORM+질환 동시 표기를 유지합니다.

기본 정책은 **SCP 코드의 존재 여부**입니다. likelihood 0도 코드가 존재하면 라벨에 포함합니다.
문자열 사전은 실행 기능이 없는 `ast.literal_eval`로 파싱합니다.
진단 superclass가 없는 기록은 all-zero 라벨과 `label_valid=False`로 표시합니다.


In [ ]:
diagnostic = statements.loc[
    statements["diagnostic"].eq(1) & statements["diagnostic_class"].isin(CLASSES)
]
code_to_class = diagnostic["diagnostic_class"].to_dict()

def parse_codes(text):
    value = ast.literal_eval(text)
    if not isinstance(value, dict):
        raise ValueError("scp_codes must be a dictionary.")
    for key, likelihood in value.items():
        if not isinstance(key, str) or not key.strip():
            raise ValueError("Invalid SCP code key.")
        if isinstance(likelihood, bool) or not isinstance(likelihood, (int, float)):
            raise ValueError("SCP likelihood must be numeric.")
        if not math.isfinite(likelihood):
            raise ValueError("Non-finite SCP likelihood.")
    return value

def superclass_vector(text):
    active = {code_to_class[k] for k in parse_codes(text) if k in code_to_class}
    return [int(label in active) for label in CLASSES]

metadata = database.copy()
targets = np.asarray(metadata["scp_codes"].map(superclass_vector).tolist(), dtype=np.uint8)
metadata[CLASSES] = targets
metadata["diagnostic_label_count"] = targets.sum(axis=1)
metadata["has_diagnostic_superclass"] = targets.any(axis=1)
metadata["diagnostic_superclasses"] = [
    "|".join(c for c, enabled in zip(CLASSES, row) if enabled) for row in targets
]

print(f"Diagnostic-unlabeled ECGs: {(~metadata.has_diagnostic_superclass).sum():,}")
display(metadata[["ecg_id", "diagnostic_superclasses", *CLASSES, "has_diagnostic_superclass"]].head())


In [ ]:
class_counts = metadata[CLASSES].sum().rename("positive_ecgs").to_frame()
display(class_counts)
fig, ax = plt.subplots(figsize=(8, 3.5))
class_counts["positive_ecgs"].plot.bar(ax=ax, color="#3b82f6", rot=0)
ax.set(title="Diagnostic superclass counts (multi-label)", ylabel="ECGs")
plt.tight_layout()
plt.show()
plt.close(fig)


## 5. 환자를 격리하는 공식 train / validation / test 분할

fold **1–8은 train**, **9는 validation**, **10은 test**입니다.
ECG를 무작위로 섞어 분할하면 같은 환자의 다른 기록이 여러 split에 들어갈 수 있습니다.
공식 fold를 사용하고 환자 중복도 다시 검사합니다.

라벨 없는 ECG를 제외하려면 위 설정 셀에서 `DROP_UNLABELED=True`로 변경합니다.
기본은 SSL용으로 전부 유지합니다.


In [ ]:
def check_patient_isolation(frame):
    assert frame.ecg_id.is_unique, "Duplicate ECG IDs."
    assert frame.patient_id.notna().all(), "Missing patient ID."
    split_counts = frame.groupby("patient_id")["split"].nunique()
    assert split_counts.max() <= 1, "One patient appears in several splits."

assert metadata.strat_fold.isin(range(1, 11)).all()
metadata["split"] = np.where(metadata.strat_fold <= 8, "train",
                           np.where(metadata.strat_fold == 9, "validation", "test"))
metadata["source_record"] = metadata[SOURCE_COLUMN]
check_patient_isolation(metadata)

excluded = metadata.loc[
    ~metadata.has_diagnostic_superclass if DROP_UNLABELED else np.zeros(len(metadata), dtype=bool),
    ["ecg_id", "patient_id", "strat_fold", "split"],
].copy()
excluded["reason"] = "no_diagnostic_superclass"
if DROP_UNLABELED:
    metadata = metadata.loc[metadata.has_diagnostic_superclass].copy()

metadata = metadata.sort_values("ecg_id").reset_index(drop=True)
if LIMIT_PER_SPLIT is not None:
    metadata = metadata.groupby("split", group_keys=False).head(LIMIT_PER_SPLIT).copy()
check_patient_isolation(metadata)
split_summary = metadata.groupby("split").agg(
    ecgs=("ecg_id", "size"), patients=("patient_id", "nunique"),
    labeled_ecgs=("has_diagnostic_superclass", "sum"),
).reindex(SPLITS)
display(split_summary)


## 6. WFDB 원본 로드와 신호 품질 검사

WFDB는 `(시간, lead)`로 반환합니다. 학습용으로 lead 이름을 확인하고
`(12, 시간)`으로 바꿉니다. 단위는 mV, 길이는 10초여야 합니다.

NaN/Inf, lead 누락, 잘못된 길이는 실행 오류로 처리합니다.
flat lead와 큰 진폭은 삭제하지 않고 아래 품질 지표로 기록합니다.

- lead 표준편차 < `1e-4 mV` 또는 진폭 범위 < `1e-3 mV`: near-flat
- 최대 절댓값 > `20 mV`: extreme amplitude

기존 노이즈·pacemaker 메타데이터도 CSV에 그대로 보존합니다.


In [ ]:
def load_ecg(row):
    signal, fields = wfdb.rdsamp(str(DATA_DIR / row["source_record"]))
    names = [str(name).strip().upper() for name in fields["sig_name"]]
    expected = [name.upper() for name in LEADS]
    assert len(names) == 12 and len(set(names)) == 12 and set(names) == set(expected)
    assert float(fields["fs"]) == SOURCE_FS, "Unexpected sampling rate."
    assert all(str(unit).lower() == "mv" for unit in fields["units"])
    assert signal.shape == (SOURCE_FS * 10, 12), "Unexpected signal shape."
    raw = signal[:, [names.index(name) for name in expected]].T.astype(np.float32)
    assert np.isfinite(raw).all(), "Raw signal contains NaN/Inf."
    return raw

def quality_metrics(raw):
    std, span = raw.std(axis=1, dtype=np.float64), np.ptp(raw, axis=1)
    peak = float(np.abs(raw).max())
    return {
        "quality_near_flat_std": bool((std < 1e-4).any()),
        "quality_near_flat_range": bool((span < 1e-3).any()),
        "quality_extreme_amplitude": peak > 20,
        "signal_max_abs_mv": peak,
        "signal_min_lead_std_mv": float(std.min()),
        "signal_min_lead_range_mv": float(span.min()),
    }

example_row = metadata.iloc[0]
example_raw = load_ecg(example_row)
print(f"ECG {int(example_row.ecg_id)}: {example_raw.shape}, {SOURCE_FS} Hz")
display(pd.Series(quality_metrics(example_raw), name="raw signal quality").to_frame())


## 7. 500 → 250 Hz 리샘플링

샘플 수가 lead당 5,000개에서 2,500개로 줄어듭니다.
`resample_poly`는 anti-aliasing을 포함하므로 단순히 매 두 번째 sample을 선택하는 방식과 다릅니다.
경계는 선형 연장(`padtype="line"`)을 사용합니다. 10초 길이는 유지합니다.


In [ ]:
def resample_ecg(raw):
    if SOURCE_FS == TARGET_FS:
        return raw.copy()
    ratio = Fraction(TARGET_FS, SOURCE_FS)
    result = resample_poly(raw, ratio.numerator, ratio.denominator,
                           axis=1, padtype="line")
    assert result.shape == (12, TARGET_FS * 10)
    return result.astype(np.float32)

example_resampled = resample_ecg(example_raw)
print("Raw:", example_raw.shape, "-> resampled:", example_resampled.shape)


## 8. 0.67–50 Hz zero-phase band-pass

Butterworth 필터를 SOS 형태로 만든 뒤 `sosfiltfilt`로 앞뒤 방향으로 적용합니다.
phase 지연을 줄이는 offline 처리이며 실시간 인과적 필터와는 다릅니다.
경계에서는 SciPy의 기본 odd-extension padding을 사용합니다.

4차 band-pass 설계에 forward/backward 처리를 적용하므로 최종 주파수 응답은
한 번 통과했을 때의 응답과 다릅니다.
baseline wander를 억제하지만 ST 형태나 pacemaker spike에도 영향을 줄 수 있어
필터 사용/미사용 성능을 비교해야 합니다.


In [ ]:
sos = butter(FILTER_ORDER, [LOW_HZ, HIGH_HZ], btype="bandpass",
             fs=TARGET_FS, output="sos") if USE_BANDPASS else None

def filter_ecg(resampled):
    if sos is None:
        return resampled.copy()
    return sosfiltfilt(sos, resampled, axis=1).astype(np.float32)

example_filtered = filter_ecg(example_resampled)
print("Filtered:", example_filtered.shape, "| enabled:", USE_BANDPASS)


## 9. 정규화

기본 `per-record`는 ECG마다 각 lead의 시간축 평균과 표준편차를 계산합니다.

\[
z_{l,t} = \frac{x_{l,t} - \mu_l}{\max(\sigma_l,\epsilon)}
\]

상수 lead는 0이 되고, 작은 표준편차는 epsilon으로 보호합니다.
환자·장비의 gain 차이를 줄이지만 절대 진폭 정보를 제거합니다.
한 ECG 내부의 통계만 사용하므로 train과 test의 통계를 합치지 않습니다.

`per-lead-train`은 필터링 후 **train 데이터만** 이용해 lead별 평균·표준편차를 적합합니다.
validation/test에는 그 값을 재사용합니다. 기존 산출물이 설정과 일치하면 저장된 통계를 읽습니다.


In [ ]:
# 기존 산출물은 설정과 ECG 순서가 일치할 때만 재사용합니다.
def requested_config():
    return {
        "source_sampling_rate": SOURCE_FS, "target_sampling_rate": TARGET_FS,
        "bandpass_low": LOW_HZ if USE_BANDPASS else None,
        "bandpass_high": HIGH_HZ if USE_BANDPASS else None,
        "filter_order": FILTER_ORDER, "normalization": NORMALIZATION,
        "epsilon": EPS, "drop_unlabeled": DROP_UNLABELED,
        "min_likelihood": None, "limit_per_split": LIMIT_PER_SPLIT,
    }

def find_existing_artifact():
    if not OUTPUT_DIR.exists():
        return None
    path = OUTPUT_DIR / "manifest.json"
    if not path.exists():
        raise ValueError("Output folder exists without a manifest. Choose a new OUTPUT_DIR.")
    saved = json.loads(path.read_text(encoding="utf-8"))
    differences = {key: (saved["config"].get(key), value)
                   for key, value in requested_config().items()
                   if saved["config"].get(key) != value}
    if differences:
        raise ValueError(f"Settings differ from existing output: {differences}. Choose a new OUTPUT_DIR.")
    for split in SPLITS:
        old = pd.read_csv(OUTPUT_DIR / f"metadata_{split}.csv", low_memory=False)
        current = metadata.loc[metadata.split.eq(split)].sort_values("ecg_id")
        if old.ecg_id.tolist() != current.ecg_id.tolist():
            raise ValueError("ECG selection differs. Choose a new OUTPUT_DIR.")
    return saved

existing_manifest = find_existing_artifact()
train_mean = train_std = None

if NORMALIZATION == "per-lead-train":
    if existing_manifest is not None:
        saved_stats = json.loads((OUTPUT_DIR / "normalization.json").read_text(encoding="utf-8"))
        train_mean, train_std = np.asarray(saved_stats["lead_mean"]), np.asarray(saved_stats["lead_std"])
    else:
        # ECG별 평균·분산을 결합해 train 전체의 통계를 안정적으로 계산합니다.
        count, mean, m2 = 0, np.zeros(12), np.zeros(12)
        train_rows = metadata.loc[metadata.split.eq("train")]
        for number, (_, row) in enumerate(train_rows.iterrows(), 1):
            x = filter_ecg(resample_ecg(load_ecg(row))).astype(np.float64)
            n = x.shape[1]
            batch_mean = x.mean(axis=1)
            delta = batch_mean - mean
            new_count = count + n
            m2 += ((x - batch_mean[:, None]) ** 2).sum(axis=1) + delta ** 2 * count * n / new_count
            mean += delta * n / new_count
            count = new_count
            if number % PROGRESS_EVERY == 0:
                print(f"Train statistics: {number:,}/{len(train_rows):,}", flush=True)
        assert count > 0
        train_mean, train_std = mean, np.maximum(np.sqrt(m2 / count), EPS)

def normalize_ecg(filtered):
    if NORMALIZATION == "none":
        return filtered.astype(np.float32)
    if NORMALIZATION == "per-record":
        mean = filtered.mean(axis=1, keepdims=True, dtype=np.float64)
        std = filtered.std(axis=1, keepdims=True, dtype=np.float64)
    else:
        mean, std = train_mean[:, None], train_std[:, None]
    return ((filtered - mean) / np.maximum(std, EPS)).astype(np.float32)

def preprocess_ecg(raw):
    result = normalize_ecg(filter_ecg(resample_ecg(raw)))
    assert result.shape == (12, TARGET_FS * 10) and np.isfinite(result).all()
    return result

example_processed = normalize_ecg(example_filtered)
display(pd.DataFrame({
    "lead": LEADS, "mean": example_processed.mean(axis=1),
    "std": example_processed.std(axis=1),
}))
print("Existing artifact:", "reuse" if existing_manifest is not None else "new build")


## 10. 실제 ECG 파형으로 전처리 효과 보기

lead II에서 원본·리샘플링·필터링·정규화를 비교합니다.
마지막 패널의 세로축은 mV가 아닌 정규화 값입니다.
각 단계의 길이와 값 변화를 함께 확인하세요.


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
lead_index = LEADS.index("II")
stages = [
    (example_raw, SOURCE_FS, f"Raw {SOURCE_FS} Hz", "mV"),
    (example_resampled, TARGET_FS, f"Resampled {TARGET_FS} Hz", "mV"),
    (example_filtered, TARGET_FS, "Band-pass" if USE_BANDPASS else "Filter off", "mV"),
    (example_processed, TARGET_FS, f"Normalized: {NORMALIZATION}",
     "mV" if NORMALIZATION == "none" else "scaled"),
]
for ax, (x, fs, title, unit) in zip(axes, stages):
    ax.plot(np.arange(x.shape[1]) / fs, x[lead_index], lw=0.8)
    ax.set(title=title, ylabel=unit)
axes[-1].set_xlabel("Time (seconds)")
fig.suptitle(f"ECG {int(example_row.ecg_id)} — lead II")
plt.tight_layout()
plt.show()
plt.close(fig)


## 11. split별 학습 데이터 저장 또는 기존 데이터 불러오기

큰 신호는 `open_memmap`으로 레코드별로 저장하여 전체 배열을 RAM에 올리지 않습니다.
split 내부는 **ecg_id 오름차순**이며 CSV의 `artifact_index`가 NPY 배열의 행 번호입니다.

| 파일 | 용도 |
| --- | --- |
| signals_train/validation/test.npy | float32, (N, 12, 2500) 신호 |
| labels_train/validation/test.npy | uint8, (N, 5) multi-hot 라벨 |
| label_valid_train/validation/test.npy | bool, (N,) 지도학습 라벨 유효성 |
| metadata_train/validation/test.csv | ECG/환자 ID, 원본 메타데이터, 품질 플래그 |
| manifest.json | 설정, shape, dtype, 원본 CSV checksum |
| normalization.json | 정규화 방식, train 통계(해당 모드일 때) |
| classes.json / lead_names.json | 클래스·lead 순서 |
| scp_to_superclass.json | 코드 mapping |
| label_prevalence.csv | split별 클래스 빈도 |
| excluded_records.csv | 제외 ID·사유; 기본은 빈 표 |

전체 신호의 크기는 약 2.62 GB입니다.
신규 빌드는 임시 폴더에 기록하고 검증 후 최종 경로로 옮깁니다.
실패한 신규 빌드는 진단을 위해 임시 폴더를 남깁니다.


먼저 checksum과 저장된 배열을 검증하는 함수를 정의합니다. 전체 신호를 128개 단위로 검사하고, CSV의 ID·라벨과 비교합니다.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def validate_output(folder):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    assert manifest["lead_order"] == LEADS and manifest["class_order"] == CLASSES
    for name, key in [("ptbxl_database.csv", "ptbxl_database_sha256"),
                      ("scp_statements.csv", "scp_statements_sha256")]:
        assert manifest["source"][key] == sha256_file(DATA_DIR / name), "Source CSV checksum changed."
    all_metadata, report = [], []
    for split in SPLITS:
        expected = metadata.loc[metadata.split.eq(split)].sort_values("ecg_id")
        frame = pd.read_csv(folder / f"metadata_{split}.csv", low_memory=False)
        x = np.load(folder / f"signals_{split}.npy", mmap_mode="r")
        y = np.load(folder / f"labels_{split}.npy", mmap_mode="r")
        valid = np.load(folder / f"label_valid_{split}.npy", mmap_mode="r")
        n = len(frame)
        assert x.shape == (n, 12, TARGET_FS * 10) and x.dtype == np.float32
        assert y.shape == (n, 5) and y.dtype == np.uint8
        assert valid.shape == (n,) and valid.dtype == np.bool_
        assert frame.ecg_id.tolist() == expected.ecg_id.tolist()
        assert np.array_equal(frame.patient_id.to_numpy(), expected.patient_id.to_numpy())
        assert np.array_equal(frame.strat_fold.to_numpy(), expected.strat_fold.to_numpy())
        assert frame.split.eq(split).all()
        assert np.array_equal(frame.artifact_index, np.arange(n))
        assert np.isin(y, [0, 1]).all()
        assert np.array_equal(y, frame[CLASSES].to_numpy(dtype=np.uint8))
        assert np.array_equal(y, expected[CLASSES].to_numpy(dtype=np.uint8))
        assert np.array_equal(valid, y.any(axis=1))
        assert manifest["splits"][split]["count"] == n
        # 표본이 아닌 모든 신호를 작은 청크로 검사합니다.
        mean_error, std_error, constant_leads = 0.0, 0.0, 0
        for start in range(0, n, 128):
            batch = np.asarray(x[start:start + 128])
            assert np.isfinite(batch).all(), f"Non-finite output in {split}."
            if NORMALIZATION == "per-record":
                means, stds = batch.mean(axis=2), batch.std(axis=2)
                assert np.abs(means).max() < 2e-4
                assert stds.max() < 1.0002
                mean_error = max(mean_error, float(np.abs(means).max()))
                std_error = max(std_error, float(np.maximum(stds - 1, 0).max()))
                constant_leads += int((stds < 1e-5).sum())
        report.append({"split": split, "ECGs": n, "shape": str(x.shape),
                       "finite": True, "max_abs_mean": mean_error if NORMALIZATION == "per-record" else None,
                       "constant_leads": constant_leads})
        all_metadata.append(frame)
        del x, y, valid
    check_patient_isolation(pd.concat(all_metadata, ignore_index=True))
    return pd.DataFrame(report)



아래 함수는 신규 전처리에만 사용합니다. 원본을 한 ECG씩 읽어 split별 파일에 기록하고, 저장 설정을 manifest에 남깁니다.

In [ ]:
def build_new_output():
    OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    assert not OUTPUT_DIR.exists(), "Choose a new output directory."
    work_dir = OUTPUT_DIR.with_name("." + OUTPUT_DIR.name + ".building-" + uuid.uuid4().hex[:8])
    work_dir.mkdir()
    started = time.perf_counter()
    split_manifest = {}
    print("New build directory:", work_dir, flush=True)

    write_json(work_dir / "normalization.json", {
        "mode": NORMALIZATION, "epsilon": EPS,
        "fit_split": "train" if NORMALIZATION == "per-lead-train" else None,
        "lead_mean": train_mean.tolist() if train_mean is not None else None,
        "lead_std": train_std.tolist() if train_std is not None else None,
    })
    write_json(work_dir / "classes.json", {"classes": CLASSES})
    write_json(work_dir / "lead_names.json", {"leads": LEADS})
    write_json(work_dir / "scp_to_superclass.json", code_to_class)
    excluded.to_csv(work_dir / "excluded_records.csv", index=False)
    prevalence_rows = []
    for split in [*SPLITS, "all"]:
        subset = metadata if split == "all" else metadata.loc[metadata.split.eq(split)]
        for label in CLASSES:
            prevalence_rows.append({"split": split, "class": label,
                                    "positive_count": int(subset[label].sum()),
                                    "total_count": len(subset),
                                    "prevalence": float(subset[label].mean()) if len(subset) else 0.0})
    pd.DataFrame(prevalence_rows).to_csv(work_dir / "label_prevalence.csv", index=False)

    for split in SPLITS:
        frame = metadata.loc[metadata.split.eq(split)].sort_values("ecg_id").reset_index(drop=True)
        n = len(frame)
        if n == 0:
            raise ValueError(f"No records selected for {split}.")
        shape = (n, 12, TARGET_FS * 10)
        x = open_memmap(work_dir / f"signals_{split}.npy", mode="w+", dtype=np.float32, shape=shape)
        y = frame[CLASSES].to_numpy(dtype=np.uint8)
        valid = y.any(axis=1)
        np.save(work_dir / f"labels_{split}.npy", y, allow_pickle=False)
        np.save(work_dir / f"label_valid_{split}.npy", valid, allow_pickle=False)
        qualities = []
        for index, row in frame.iterrows():
            raw = load_ecg(row)
            qualities.append(quality_metrics(raw))
            x[index] = preprocess_ecg(raw)
            if (index + 1) % PROGRESS_EVERY == 0 or index + 1 == n:
                print(f"{split}: {index + 1:,}/{n:,}", flush=True)
        x.flush()
        del x
        frame.insert(0, "artifact_index", np.arange(n))
        quality = pd.DataFrame(qualities)
        frame = pd.concat([frame, quality], axis=1)
        frame.to_csv(work_dir / f"metadata_{split}.csv", index=False)
        split_manifest[split] = {
            "count": n,
            "signals": {"file": f"signals_{split}.npy", "shape": list(shape), "dtype": "float32",
                        "bytes": (work_dir / f"signals_{split}.npy").stat().st_size},
            "labels": {"file": f"labels_{split}.npy", "shape": [n, 5], "dtype": "uint8"},
            "label_valid": {"file": f"label_valid_{split}.npy", "shape": [n], "dtype": "bool",
                            "valid_count": int(valid.sum())},
            "metadata": f"metadata_{split}.csv",
            "quality_flags": {"near_flat_std": int(quality.quality_near_flat_std.sum()),
                              "near_flat_range": int(quality.quality_near_flat_range.sum()),
                              "extreme_amplitude": int(quality.quality_extreme_amplitude.sum())},
        }
    source_checksums = {
        "data_dir": str(DATA_DIR),
        "ptbxl_database_sha256": sha256_file(DATA_DIR / "ptbxl_database.csv"),
        "scp_statements_sha256": sha256_file(DATA_DIR / "scp_statements.csv"),
    }
    manifest = {
        "schema_version": 1, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset": "PTB-XL",
        "dataset_version": "1.0.3" if (DATA_DIR / "ptbxl_v103_changelog.txt").exists() else "unknown",
        "source": source_checksums,
        "config": {**requested_config(), "data_dir": str(DATA_DIR), "output_dir": str(OUTPUT_DIR)},
        "lead_order": LEADS, "class_order": CLASSES,
        "label_policy": {"type": "diagnostic-superclass-multilabel", "scp_code_policy": "key-presence",
                         "unlabeled_records": "drop" if DROP_UNLABELED else "retain-with-mask"},
        "split_policy": {"train": "strat_fold 1-8", "validation": "strat_fold 9", "test": "strat_fold 10",
                         "patient_overlap_checked": True},
        "preprocessing_order": ["load-physical-mV", "canonical-lead-reorder",
                                "polyphase-resample" if SOURCE_FS != TARGET_FS else "no-resample",
                                "zero-phase-butterworth-bandpass" if USE_BANDPASS else "no-filter",
                                NORMALIZATION],
        "excluded_records": len(excluded), "splits": split_manifest,
        "elapsed_seconds": time.perf_counter() - started,
    }
    write_json(work_dir / "manifest.json", manifest)
    validate_output(work_dir)
    work_dir.rename(OUTPUT_DIR)
    return manifest


이 셀이 실제 실행 지점입니다. 설정이 같은 기존 데이터가 있으면 읽고, 없으면 새로 생성합니다. 검증 결과는 아래 표에 표시합니다.

In [ ]:

if existing_manifest is None:
    manifest = build_new_output()
else:
    manifest = existing_manifest
    print("Reusing preprocessed arrays:", OUTPUT_DIR)

validation_report = validate_output(OUTPUT_DIR)
display(validation_report)
print("All ECGs checked: finite values, label alignment, patient isolation.")


## 12. 학습에서 데이터를 읽는 방법

`np.load(..., mmap_mode="r")`로 필요한 기록만 읽습니다.
한 ECG는 `(12, 2500)`, 한 라벨은 클래스 순서에 따른 `(5,)`입니다.

**SSL:** 라벨 유무와 관계없이 train 신호를 사용합니다.
**지도학습:** `label_valid=True`인 기록에만 loss를 적용합니다.
train/validation/test의 역할은 학습 코드에서도 유지하세요.


In [ ]:
X_train = np.load(OUTPUT_DIR / "signals_train.npy", mmap_mode="r")
y_train = np.load(OUTPUT_DIR / "labels_train.npy", mmap_mode="r")
label_valid_train = np.load(OUTPUT_DIR / "label_valid_train.npy", mmap_mode="r")
train_metadata = pd.read_csv(OUTPUT_DIR / "metadata_train.csv", low_memory=False)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("Supervised eligible train ECGs:", int(label_valid_train.sum()))
display(train_metadata[["artifact_index", "ecg_id", "patient_id", "diagnostic_superclasses"]].head())

# 학습 mini-batch 예시: 전체 supervised 배열을 복사하지 않고 필요한 index만 선택
eligible_indices = np.flatnonzero(label_valid_train)
batch_indices = eligible_indices[:4]
X_batch = np.asarray(X_train[batch_indices])
y_batch = np.asarray(y_train[batch_indices])
print("Example mini-batch:", X_batch.shape, y_batch.shape)


## 13. 품질 플래그 확인과 산출물 설명

flat lead나 큰 진폭은 품질 검토 대상이며 자동 제외 기준은 아닙니다.
품질을 기준으로 제외하는 실험에서는 규칙을 먼저 정하고 manifest와 split별 변화를 기록하세요.
아래 표는 실제 저장된 데이터에서 계산하므로 새 설정·새 데이터에도 대응합니다.

`artifacts/`의 파일들은 학습 데이터이고, 이 노트북은 그 파일들을 만들고 읽는 실행 문서입니다.
원본 ECG 폴더와 학습 산출물은 Git에 올리지 않습니다.


In [ ]:
quality_summary, flagged_frames = [], []
for split in SPLITS:
    frame = pd.read_csv(OUTPUT_DIR / f"metadata_{split}.csv", low_memory=False)
    flags = ["quality_near_flat_std", "quality_near_flat_range", "quality_extreme_amplitude"]
    quality_summary.append({"split": split, "ECGs": len(frame),
                            **{flag: int(frame[flag].sum()) for flag in flags}})
    flagged_frames.append(frame.loc[frame[flags].any(axis=1),
                                   ["ecg_id", "split", *flags, "signal_max_abs_mv"]])
display(pd.DataFrame(quality_summary))
display(pd.concat(flagged_frames, ignore_index=True))
display(pd.DataFrame([
    {"file": path.name, "MB": round(path.stat().st_size / 1_000_000, 3)}
    for path in sorted(OUTPUT_DIR.iterdir()) if path.is_file()
]))


## 14. 비교 실험을 할 때 바꿀 설정

| 목적 | 변경 |
| --- | --- |
| 필터 영향 | USE_BANDPASS = False |
| 절대 진폭 보존 | NORMALIZATION = "none" |
| train 통계로 정규화 | NORMALIZATION = "per-lead-train" |
| 500 Hz 유지 | TARGET_FS = 500 |
| 공식 100 Hz 배포본 | SOURCE_FS = 100; TARGET_FS = 100 |
| 지도학습 기록만 저장 | DROP_UNLABELED = True |

설정 변경 후 `OUTPUT_DIR`을 새로운 이름으로 바꾸고 커널을 재시작해 위에서 아래로 실행합니다.
기존 산출물과 설정이 다르면 덮어쓰지 않고 오류로 알려줍니다.
모든 lead를 독립적으로 정규화하면 lead 사이의 진폭 비율이나 Einthoven 관계가 달라질 수 있습니다.
masked reconstruction 결과는 lead 간 선형 관계에 의한 쉬운 복원과 downstream 성능을 함께 검토하세요.

참고: [PTB-XL v1.0.3](https://physionet.org/content/ptb-xl/1.0.3/),
[데이터셋 논문](https://doi.org/10.1038/s41597-020-0495-6),
[WFDB 읽기 문서](https://wfdb.readthedocs.io/en/latest/io.html#wfdb.io.rdsamp).

끝까지 실행했으면 전처리된 배열과 검증 결과를 학습 단계에서 사용할 수 있습니다.
